**Feature pruning** is the model-free, post-hoc reduction of a ``df_feat`` (here from :meth:`CPP.run`) before model-based selection. :meth:`SequenceFeature.prune_by_variance` drops near-constant features — those whose feature values barely change across the samples — measured on the feature matrix that :meth:`SequenceFeature.feature_matrix` builds from ``df_parts``. The recommended order is **variance -> correlation -> select_features**.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=20)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
df_feat = aa.CPP(df_parts=df_parts).run(labels=labels, n_filter=50)
print(f"features from CPP.run: {len(df_feat)}")

features from CPP.run: 50


With the default ``threshold=0.0`` only strictly constant features are removed. The result is a row-filtered ``df_feat`` with the same schema:

In [2]:
df_var = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, threshold=0.0)
print(f"kept {len(df_var)} of {len(df_feat)} features")
df_var.head()

kept 50 of 50 features


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.022853,"23,27"
1,"TMD_C_JMD_C-Pattern(N,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.045706,"24,28"
2,"TMD_C_JMD_C-Pattern(N,4,8,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized frequency of turn (Crawford et al.,...",0.431,0.251,-0.251,0.088,0.143,0.000003,0.011128,"24,28,32"
3,"TMD_C_JMD_C-Pattern(N,4,8,12)-MUNV940102",Energy,Free energy (folding),Free energy (α-helix),Free energy in alpha-helical region (Munoz-Ser...,0.422,0.148,-0.148,0.056,0.097,0.000005,0.009366,"24,28,32"
4,"TMD_C_JMD_C-PeriodicPattern(C,i+4/4,2)-BEGF750103",Conformation,β-turn,β-turn,Conformational parameter of beta-turn (Beghin-...,0.409,0.139,-0.139,0.098,0.094,0.000010,0.011310,"23,27,31,35,39"


Raise ``threshold`` to prune low-variance features more aggressively. The feature matrix can also be supplied directly via ``X`` (e.g. to reuse a matrix across several pruning calls, or to prune a :meth:`CPP.run_num` table); then ``df_parts`` / ``df_scales`` are ignored. ``accept_gaps`` and ``n_jobs`` are forwarded to the matrix build, and a custom ``df_scales`` can be passed:

In [3]:
X = sf.feature_matrix(features=df_feat, df_parts=df_parts, accept_gaps=False, n_jobs=1)
df_var_strict = sf.prune_by_variance(df_feat=df_feat, X=X, threshold=0.01)
print(f"threshold=0.01 kept {len(df_var_strict)} of {len(df_feat)} features")

threshold=0.01 kept 48 of 50 features


**What can go wrong?** A ``threshold`` above every feature's variance would remove all features, so it raises a ``ValueError`` rather than returning an empty table:

In [4]:
try:
    sf.prune_by_variance(df_feat=df_feat, X=X, threshold=1e6)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: 'threshold' (1000000.0) removed all features. Lower it to retain features.
